# Evaluación del Modelo Final

Análisis detallado de errores: ¿qué tipo de eventos generan más falsos positivos?

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from src.data.loader import load_bgl_logs
from src.data.preprocessor import add_severity_score, train_test_split_temporal
from src.features.engineering import build_features, load_features, save_features
from src.models.detector import AnomalyDetector
from src.models.evaluator import evaluate, get_pr_curve

plt.rcParams['figure.dpi'] = 120

MODEL_PATH = Path('../models/saved/lof_v1.joblib')
FEATURES_PATH = Path('../data/processed/features_train.parquet')

In [ ]:
detector = AnomalyDetector.load(MODEL_PATH)
df_feat = load_features(FEATURES_PATH)

feature_cols = [c for c in df_feat.columns if c not in {'timestamp', 'node', 'is_anomaly'}]
_, test_df = train_test_split_temporal(df_feat, test_fraction=0.2)

X_test = test_df[feature_cols].fillna(0).values
y_test = test_df['is_anomaly'].values
y_pred = detector.predict(X_test)
scores = detector.score_samples(X_test)

test_df = test_df.copy()
test_df['y_pred'] = y_pred
test_df['anomaly_score'] = scores

## 1. Matriz de confusión

In [ ]:
from sklearn.metrics import confusion_matrix

cm = confusion_matrix(y_test, y_pred)
tn, fp, fn, tp = cm.ravel()

print('=== Matriz de Confusión ===')
print(f'              Pred Normal  Pred Anomalía')
print(f'Real Normal      {tn:>8,}       {fp:>8,}  ← Falsos Positivos')
print(f'Real Anomalía    {fn:>8,}       {tp:>8,}  ← Verdaderos Positivos')
print()
print(f'Precision: {tp/(tp+fp):.4f}  (de las alertas, cuántas son reales)')
print(f'Recall:    {tp/(tp+fn):.4f}  (de las anomalías reales, cuántas detectamos)')

## 2. Análisis de Falsos Positivos

In [ ]:
# Recargar datos raw para cruzar con nivel/componente
df_raw = load_bgl_logs(Path('../data/raw/BGL.log'), nrows=500_000)
df_raw = add_severity_score(df_raw)
_, df_test_raw = train_test_split_temporal(df_raw)
df_test_raw = df_test_raw.reset_index(drop=True)
df_test_raw = df_test_raw.iloc[:len(test_df)].copy()
df_test_raw['y_pred'] = y_pred
df_test_raw['anomaly_score'] = scores

false_positives = df_test_raw[(df_test_raw['y_pred'] == 1) & (~df_test_raw['is_anomaly'])]
print(f'Total falsos positivos: {len(false_positives):,}')

print('\nFP por nivel de log:')
print(false_positives['level'].value_counts())

print('\nFP por componente (top 10):')
print(false_positives['component'].value_counts().head(10))

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# FP por nivel
fp_levels = false_positives['level'].value_counts().head(8)
fp_levels.plot(kind='bar', ax=ax1, color='#FF9800')
ax1.set_title('Falsos Positivos por nivel de log')
ax1.set_xlabel('Nivel')
ax1.set_ylabel('Cantidad')
ax1.tick_params(axis='x', rotation=45)

# FP por score distribution
ax2.hist(false_positives['anomaly_score'], bins=50, color='#FF9800', alpha=0.8, label='FP')
ax2.hist(df_test_raw[(df_test_raw['y_pred'] == 1) & (df_test_raw['is_anomaly'])]['anomaly_score'],
         bins=50, color='#F44336', alpha=0.7, label='TP')
ax2.set_xlabel('Anomaly score')
ax2.set_ylabel('Cantidad')
ax2.set_title('Score distribution: FP vs TP')
ax2.legend()

plt.tight_layout()
plt.savefig('../reports/figures/04_false_positives.png', dpi=150)
plt.show()

## 3. Ejemplos de anomalías detectadas (con score alto)

In [ ]:
top_anomalies = df_test_raw[df_test_raw['y_pred'] == 1].nlargest(10, 'anomaly_score')
pd.set_option('display.max_colwidth', 80)
top_anomalies[['timestamp', 'node', 'level', 'component', 'content', 'anomaly_score', 'is_anomaly']].head(10)